# Selectividad Exams Parser

This script reads the content of the PDF files, to extract the information from files

## Libraries import

### Required packages for OCR processing:
**Ubuntu based:**
- tesseract-ocr
- tesseract-ocr-spa
- poppler-utils

**Archlinux based:**
- tesseract
- tesseract-data-spa

In [21]:
# !pip install pdf2image
# !pip install pytesseract

In [3]:
from pdf2image import convert_from_path
import pytesseract
import pandas as pd
from pathlib import Path
from multiprocessing import Pool
import os
import requests
from datetime import datetime

In [13]:
# set root path
root = "/home/daniel/code/emestrada_parser/"
os.chdir(root)

## Functions

### *pdf_to_text*

Extract the text from pdf files, page by page

In [ ]:
def pdf_to_text(file_path):
    """
    Processes a single PDF file to extract its content as text.
    """
    images = convert_from_path(file_path)[1:]
    # parse each page
    pages = []
    for i, page in enumerate(images):
        pages.append(pytesseract.image_to_string(page, lang='spa'))
    return pages

### *extract_statement*
Extract the statement from a page in text format

In [5]:
def extract_statement(page):
    # split the content into lines
    text = page.split('\n')

    # list to store the processed lines
    final_text = []
    # start from the 4th line
    for line in text[4:]:

        # check the first word of the line
        first_word = line.replace('.', '').split(' ')[0].strip().upper()

        # stop at examen details line
        if first_word in ['SOCIALES', 'FISICA', 'MATES',
                          'QUÍMICA', 'QUIMICA', 'QUIÍMICA']:
            final_text.append(line.strip())
            break
        
        # add the line to final_text
        else:
            if not line == '':
                final_text.append(line.strip())

    # join the list into a single string
    final_text = '\n'.join(final_text)

    return final_text

### *extract_exam_details*
Extract the exam details, like subject, year, exam and exercise number.

In [6]:
def extract_exam_details(statement):
    # get last line with exam details
    lines = [i for i in statement.split('\n') if i != '']
    exam_details = lines[-1].lower()

    # strip whitespaces
    exam_details = (exam_details.replace('.', '')            
                    .replace(',', '')
                    .replace('(', '')
                    .replace(')', '').strip()
    )
    
    # split the words by ' '
    exam_details = exam_details.split(' ')

    # MATES CCSS
    # SOCIALES II. 2017 JUNIO. EJERCICIO 2. OPCIÓN A
    # 
    if exam_details[0].startswith('sociales'):
        exam_details[0] = 'MATES_CCSS'
        # remove the second element in the list
        del exam_details[1]
    
       
    # exam dictionary
    details = ['subject', 'year', 'exam', 'exercise']
    exam_dict = dict.fromkeys(details)
    exam_dict = {k:None for k in details}

    ## fill dictionary with values
    
    # subject
    exam_dict['subject'] = exam_details[0]
    
    # year
    # SOCIALES II. PONENCIA 2009. EJERCICIO 2
    if exam_details[1] == 'ponencia':
        # switch ponencia and year
        exam_details[1], exam_details[2] = exam_details[2], exam_details[1]

    exam_dict['year'] = int(exam_details[1])

    # parse the exam string
    if exam_details[2].startswith('reserva'):
        exam_dict['exam'] = ' '.join(exam_details[2:4]).title()
        exam_index_start = 4
    else:
        exam_dict['exam'] = exam_details[2].title()
        exam_index_start = 3
    
    exam_dict['exercise'] = ' '.join(exam_details[exam_index_start:]).title()

    
    return exam_dict

### *generate_error_log*
generate a error log and add it to errors_timestamp.log file

In [7]:
def generate_error_log(file, page, error_type, statement = None, details = None):
    log = f'Error processing {error_type}, in {file.name} at page {page+2}\n'
    
    # add timestamp to log file
    current_datetime = datetime.now()
    formatted_dt = current_datetime.strftime("%d-%m-%Y")
    with open(f'./error_logs/errors_{formatted_dt}.log', 'a') as f:
        f.write(log)
        f.write('*' * 10 + '\n')
        if statement:
            f.write(f'{statement}\n')
        elif details:
            f.write(f'{details}\n')
        f.write('*' * 10 + '\n')
    
    # print log to console without the new line
    print(log[:-2])

### *process_file*
Process a pdf files, extracting its information: statement and exam information 

In [ ]:
def process_file(file: Path) -> dict:
    exercises_dict = {}
    topic = file.stem.split(' - ')[-1]   
    
    # lists to store the values
    subjects = []
    years = []
    exams = []
    exercises = []
    statements = []
    pages = pdf_to_text(file)

    # parse each page in the pdf file

    output = f'Success parsing file: {file.stem}'
    
    for index, page in enumerate(pages):

        # extract statement from page
        try:
            statement = extract_statement(page)
        # error with extracting statement
        except:
            for line in statement.split('\n'):
                if line.startswith('www.emestrada.org'):
                    continue
            generate_error_log(file = file, page = index, error_type= 'statement', statement=page)
            continue

        # extract exam details
        try:
            details = extract_exam_details(statement)
            subjects.append(details['subject'])
            years.append(details['year'])
            exams.append(details['exam'])
            exercises.append(details['exercise'])
            statements.append(statement)
        # error with extracting details
        except:
            # pages ending with www.emestrada.org are pages with solution to exercise
            if not statement.endswith('www.emestrada.org'):
                generate_error_log(file = file, page = index, error_type= 'details', details = statement)
                output = f'File {file.stem} parsed but with errors'
            continue
        
    print(output)

    # generate the key and content for the exercises dictionary
    if details:
        key = details['subject'] + ' ' + str(details['year']) + ' ' + topic
        exercises_dict[key] = {
            'subject' : subjects,
            'year' : years,
            'topic' : [topic] * len(subjects),
            'exam' : exams,
            'exercise' : exercises,
            'statement' : statements
        }
        return exercises_dict

In [57]:
dic = process_file("./test/2000 - Equilibrio Químico.pdf")

Success parsing file: 2000 - Equilibrio Químico


### *create_content_dict*
Creates a python dictionary with the exam details and its statement

In [9]:
def create_content_dict():
    # create the dict from keys
    keys = ['subject', 'year', 'topic', 'exam', 'exercise', 'statement']
    #content = dict.fromkeys(exercises_dict[first_processed_file].keys())
    content = dict.fromkeys(keys)
    # fill the dict with empty lists
    for key in content.keys():
        content[key] = []

    return content

### *process_folder*
Processes all files within a folder

In [31]:
def process_folder(folder:str) -> pd.DataFrame:
    df = pd.DataFrame()
    
    # convert into path
    folder_path = root / Path(folder)
    # get the list of pdf files within the folder
    files = sorted( [i for i in Path(folder_path).iterdir()
                        if i.suffix == '.pdf'] )
    
    # process each pdf file to extract its information
    for file in files:
        try:
            exercises_dict = process_file(file)
            # each year contains a dictionary with keys containing lists of values
            for year, dict_ in exercises_dict.items():
                df = pd.concat([df, pd.DataFrame(dict_)], axis = 0)

        except:
            pass
        
    # correct wrong subjects
    #df.subject = df.subject.apply(lambda x: 'Química'
    #                                if x.endswith('mica') else x)
    
    # create csv folder to store csv files
    ## concatenate folder path provided with a subfolder called csv_files
    csv_folder = folder_path / Path('csv_files')
    if not csv_folder.exists():
        csv_folder.mkdir()
        print(f"Folder '{csv_folder}' created.")
    else:
        print(f"Folder '{csv_folder}' already exists.")
    # export the output of the processed pdf files to a csv file
    output_file = csv_folder / Path(f'exercises_{folder_path.stem}.csv')
    
    df.to_csv(output_file, index=False)
    
    return df

In [32]:
process_folder("pdf_files/quimica/solubilidad")

Success parsing file: 2010 - Solubilidad
Success parsing file: 2011 - Solubilidad
Success parsing file: 2012 - Solubilidad
Success parsing file: 2013 - Solubilidad
Success parsing file: 2014 - Solubilidad
Success parsing file: 2015 - Solubilidad
Success parsing file: 2016 - Solubilidad
Success parsing file: 2017 - Solubilidad
Success parsing file: 2018 - Solubilidad
Success parsing file: 2019 - Solubilidad
Success parsing file: 2020 - Solubilidad
Success parsing file: 2021 - Solubilidad
Success parsing file: 2022 - Solubilidad
Success parsing file: 2024 - Solubilidad
Success parsing file: 2025 - Solubilidad
Folder '/home/daniel/code/emestrada_parser/pdf_files/quimica/solubilidad/csv_files' created.


,subject,year,topic,exam,exercise,statement
0,química,2010,Solubilidad,Junio,Ejercicio 3 Opción A,Se dispone de una disolución acuosa saturada d...
1,quiímica,2010,Solubilidad,Reserva 2,Ejercicio 3 Opcion A,Los productos de solubilidad del cloruro de pl...
2,química,2010,Solubilidad,Reserva 3,Ejercicio 5 Opción B,"A 25 *C la solubilidad del PbI, en agua pura e..."
3,química,2010,Solubilidad,Reserva 4,Ejercicio 6 Opción A,A 25%C el producto de solubilidad en agua del ...
0,química,2011,Solubilidad,Reserva 1,Ejercicio 3 Opción A,El hidróxido de magnesio es un compuesto poco ...
...,...,...,...,...,...,...
5,química,2022,Solubilidad,Reserva 3,Ejercicio C2,"a) La solubilidad del hidróxido de cobre(II), ..."
6,química,2022,Solubilidad,Reserva 4,Ejercicio C4,a) En 200 mL de una disolución saturada de hid...
7,química,2022,Solubilidad,Julio,Ejercicio C2,a) Sabiendo que en 200 mL de una disolución sa...
0,química,2024,Solubilidad,Junio,C2,Para preparar 250 mL de disolución saturada de...


## Process pdf files

In [33]:
folders = sorted([i for i in Path('./pdf_files/').iterdir() if i.is_dir()])
folders = folders[:-1]
folders

FileNotFoundError: [Errno 2] No such file or directory: 'pdf_files'

In [ ]:
def process_all_files_in_folder(folder:str) -> None:
    start, start_formatted = what_time()
    send_telegram_message(f'Parsing process started at: {start_formatted}')

    subject_df = pd.DataFrame()
    folder = Path(folder)
    subfolders = sorted(folder.iterdir())
    for subfolder in subfolders:
        try:
            
            topic_df = process_folder(subfolder, return_df = True)
            print(f'Success processing folder: {subfolder.stem}')
            send_telegram_message(f'Success processing folder: {folder.stem}')
            subject_df = pd.concat([subject_df, topic_df], axis = 0)
            
        except Exception as e:
            send_telegram_message(f'Error processing folder: {subfolder.stem}' \
                                f'\n{e}')
    
    return subject_df

    end, end_formatted = what_time()

    send_telegram_message(f'Parsing process finished at: {end_formatted}'\
                        f'\nTime elapsed in minutes: {elapsed_time_minutes(start, end)}')
    send_telegram_message('✅ Your files have been processed!')

In [ ]:
# QUIMICA_df = process_all_files_in_folder('./pdf_files/QUIMICA')

In [ ]:
# for col in QUIMICA_df.columns[:-1]:
#     print(f'{col}: {QUIMICA_df[col].unique()}')

In [ ]:
# df.to_csv('./csv/QUIMICA.csv', index=False)

In [ ]:
from numpy.random import choice, seed

In [ ]:
def sample_topics(files_per_folder:int = 5) -> list:
    sample_files = []
    seed(42)
    subjects = sorted([i for i in Path('./pdf_files/').iterdir() if i.is_dir()])
    # remove samples folder
    subjects = subjects[:-1]

    sample_files = []
    for s in subjects:
        for subfolder in s.iterdir():
            print(f'Processing {subfolder.stem}')
            files = list(subfolder.glob('**/*.pdf'))
            sample_files.extend(choice(files, files_per_folder, replace = False))  

    print(f'{len(sample_files)} files sampled')

    return sample_files 
    

In [ ]:
def sample_subjects(files_per_folder:int = 3) -> list:
    sample_files = []
    seed(42)

    for f in folders:
        
        files = list(f.glob("**/*.pdf"))
        sample_files.extend(choice(files,files_per_folder, replace=False))

    print(f'Files selected: {len(sample_files)}')
    
    return sorted(sample_files)    

In [ ]:
sample_files = sample_subjects(5)

In [ ]:
sample_files = sample_topics(2)

### Save the sample files into folder

In [ ]:
import shutil

In [ ]:
os.makedirs('pdf_files/sample_files', exist_ok=True)

In [ ]:
for file in sample_files:
    shutil.copy(file, f'pdf_files/sample_files/')

### Process the sample files 

In [ ]:
df = process_folder('pdf_files/sample_files')

In [ ]:
df

In [ ]:
for col in df.columns[:-1]:
    print(f'{col}: {df[col].unique()}')

In [ ]:
pages = pdf_to_text('pdf_files/folder/2003 - Conf. Electrónica.pdf')

In [ ]:
st = extract_statement(pages[0])
print(st)

In [ ]:
extract_exam_details(st)

In [ ]:
df = pd.read_csv('csv/exercises_sample_files.csv')

## Check processing of files

1. Create path to file to process
2. Use `pdf_to_text` to process the pdf file. Store output in `pages`
3. Extract statement using `extract_statement`

In [ ]:
file = Path('pdf_files/MATES_CCSS/Contraste de Hipótesis/2016 - Contraste De Hipótesis.pdf')
pages = pdf_to_text(file)

In [ ]:
statement = extract_statement(pages[0])
statement.split('\n')[-1]

In [ ]:
details = extract_exam_details(statement)
details

In [ ]:
statement.split('\n')[-1]

## Manual Addition of exercises

#### Load csv into a dataframe

In [ ]:
df = pd.read_csv('./csv/QUIMICA/exercises_Solubilidad.csv')

In [ ]:
file = Path('pdf_files/QUIMICA/Solubilidad/2023 - Solubilidad.pdf')
pages = pdf_to_text(file)

In [ ]:
st = extract_statement(pages[3])
st

'El pH de una disolución acuosa saturada de Pb(OH), es 99 a 25"C. Basándose en la reacción\nquímica correspondiente, calcule:'

In [ ]:
QUIMICA_df

,subject,year,topic,exam,exercise,statement
0,Química,2012,Equilibrio Químico,Junio,Ejercicio 5 Opción B,"PCl;(g) Z2 PCl,(g) + Cl,(g)\nCuando se alcanza..."
1,Química,2012,Equilibrio Químico,Reserva 1,Ejercicio 6 Opción A,a) El valor de Kc.\nb) El grado de disociación...
2,Química,2012,Equilibrio Químico,Reserva 2,Ejercicio 3 Opción A,a) Aumentar la temperatura.\nb) Retirar del re...
3,Química,2012,Equilibrio Químico,Reserva 2,Ejercicio 6 Opción B,temperatura de 11 *C la presión es de 0?3 atm....
4,Química,2012,Equilibrio Químico,Reserva 3,Ejercicio 3 Opción A,"reacción en cada uno de los siguientes casos, ..."
...,...,...,...,...,...,...
2,Química,2023,Enlace Químico,Reserva 2,Ejercicio B3,"Los átomos A, B, C y D corresponden a elemento..."
3,Química,2023,Enlace Químico,Reserva 3,Ejercicio B3,Justifique si las siguientes afirmaciones son ...
4,Química,2023,Enlace Químico,Reserva 4,Ejercicio B3,Justifique si las siguientes sustancias son co...
5,Química,2023,Enlace Químico,Julio,Ejercicio B3,Responda a las siguientes cuestiones de manera...


In [ ]:
statement = '''El pH de una disolución acuosa saturada de Pb(OH)2 es 9’9 a 25ºC. Basándose en la reacción
química correspondiente, calcule:
a) La solubilidad molar en agua y el producto de solubilidad del Pb(OH) a 25ºC.
b) La solubilidad del Pb(OH)2 en una disolución de NaOH 0’1 M.'''


# SOCIALES II. 2017 JUNIO. EJERCICIO 2. OPCIÓN A
exam_details = 'QUÍMICA. 2023. RESERVA 2. EJERCICIO C2'

In [ ]:
extract_exam_details(exam_details)

{'subject': 'química',
 'year': 2023,
 'exam': 'Reserva 2',
 'exercise': 'Ejercicio C2'}

In [ ]:
df = add_row(QUIMICA_df, statement, exam_details)

In [ ]:
df

,subject,year,topic,exam,exercise,statement
0,Química,2012,Equilibrio Químico,Junio,Ejercicio 5 Opción B,"PCl;(g) Z2 PCl,(g) + Cl,(g)\nCuando se alcanza..."
1,Química,2012,Equilibrio Químico,Reserva 1,Ejercicio 6 Opción A,a) El valor de Kc.\nb) El grado de disociación...
2,Química,2012,Equilibrio Químico,Reserva 2,Ejercicio 3 Opción A,a) Aumentar la temperatura.\nb) Retirar del re...
3,Química,2012,Equilibrio Químico,Reserva 2,Ejercicio 6 Opción B,temperatura de 11 *C la presión es de 0?3 atm....
4,Química,2012,Equilibrio Químico,Reserva 3,Ejercicio 3 Opción A,"reacción en cada uno de los siguientes casos, ..."
...,...,...,...,...,...,...
3,Química,2023,Enlace Químico,Reserva 3,Ejercicio B3,Justifique si las siguientes afirmaciones son ...
4,Química,2023,Enlace Químico,Reserva 4,Ejercicio B3,Justifique si las siguientes sustancias son co...
5,Química,2023,Enlace Químico,Julio,Ejercicio B3,Responda a las siguientes cuestiones de manera...
0,Química,2024,Enlace Químico,Junio,Ejercicio B3,Dados tres elementos cuyas configuraciones ele...


In [ ]:
df.topic.fillna('Solubilidad', inplace=True)

In [ ]:
df.topic.fillna('Matrices y Determinantes', inplace=True)

In [ ]:
df.to_csv(f'/home/daniel/git_code/emestrada/csv/MATES CCSS/MATES_CCSS.csv', index=False)

## Crete combined csv file for subject

Create the pandas dataframe

In [ ]:
csv_files = [i for i in Path.iterdir(Path('./csv'))
             if i.suffix == '.csv' and not i.stem.startswith('all')]

In [ ]:
csv_files

In [ ]:
df = pd.read_csv(csv_files[0])

In [ ]:
# df = pd.read_csv(csv_files[0])
# for file in csv_files[1:]:
#     df = pd.concat([df, pd.read_csv(file)], axis = 0)
#     #df.to_csv('./csv/all_exercises.csv', index=False)

# df.sample(10)

In [ ]:
df.isna().sum()

In [ ]:
df[~df.year.isna()]

In [ ]:
df.shape

In [ ]:
df.subject.unique()

In [ ]:
df.exercise.unique()

In [ ]:
df.exercise = df.exercise.apply(lambda x: x.replace('Opcióon', 'Opción').replace('Opcion', 'Opción'))

In [ ]:
def add_comma(string):
    if len(string.split(' ')) == 4 and ',' not in string:
        words = string.split(' ')
        return f'{words[0]} {words[1]}, {words[2]} {words[3]}'
    else:
        return string

In [ ]:
df.exercise = df.exercise.apply(add_comma)

In [ ]:
df.to_csv(f'./csv/all_exercises_{df.subject.unique()[0]}.csv', index=False)

In [36]:
os.chdir("/home/daniel/code/emestrada_parser/processed_files_csv/")

In [37]:
os.getcwd()

'/home/daniel/code/emestrada_parser/processed_files_csv'

In [38]:
folder = './test'          # carpeta con los PDFs
output_csv = 'test_csv' 

In [51]:
process_folder(folder, csv_file="./eq_cinetica", return_df=True)

Success parsing file: 2000 - Equilibrio Químico
Success parsing file: 2001 - Equilibrio Químico
Success parsing file: 2002 - Equilibrio Químico
Success parsing file: 2003 - Equilibrio Químico
Success parsing file: 2004 - Equilibrio Químico
Success parsing file: 2005 - Equilibrio Químico
Success parsing file: 2006 - Equilibrio Químico
Success parsing file: 2007 - Equilibrio Químico
Success parsing file: 2008 - Equilibrio Químico
Success parsing file: 2009 - Equilibrio Químico
Success parsing file: 2010 - Equilibrio Químico
Success parsing file: 2011 - Equilibrio Químico
Success parsing file: 2012 - Equilibrio Químico
Success parsing file: 2013 - Equilibrio Químico
Success parsing file: 2014 - Equilibrio Químico
Success parsing file: 2015 - Equilibrio Químico
Success parsing file: 2016 - Equilibrio Químico
Success parsing file: 2017 - Equilibrio Químico
Success parsing file: 2018 - Equilibrio Químico
Success parsing file: 2019 - Equilibrio Químico
Success parsing file: 2020 - Equilibrio 

,subject,year,topic,exam,exercise,statement
0,química,2000,Equilibrio Químico,Junio,Ejercicio 3 Opción B,"En la tabla adjunta se recogen los valores, a ..."
1,quimica,2000,Equilibrio Químico,Reserva 1,Ejercicio 5 Opción A,"A 613 K, el valor de K, para la reacción: Fe,O..."
2,quiímica,2000,Equilibrio Químico,Reserva 1,Ejercicio 3 Opción B,Suponga el siguiente sistema en equilibrio: UO...
3,quimica,2000,Equilibrio Químico,Reserva 1,Ejercicio 4 Opción B,a) Dibuje el diagrama de entalpía teniendo en ...
4,quiímica,2000,Equilibrio Químico,Reserva 2,Ejercicio 3 Opción B,"Indique, razonadamente, si las siguientes afir..."
...,...,...,...,...,...,...
0,química,2025,Equilibrio Químico,Junio,Ejercicio 2B,La reacción química 2A + B > C tiene como ecua...
1,química,2025,Equilibrio Químico,Junio,Ejercicio 3A,"El equilibrio de descomposición del NaHCO , pu..."
2,quimica,2025,Equilibrio Químico,Reserva 1,Ejercicio 2A,Explique cómo afectan los siguientes cambios a...
3,quiímica,2025,Equilibrio Químico,Reserva 2,Ejercicio 2B,La reacción A + 2B > C es de primer orden resp...
